# Kata 20 — Preservación de Provenance

> Spec: [`specs/020-data-provenance/spec.md`](../../specs/020-data-provenance/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=10)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cada afirmación factual mantiene un mapeo tipado a su fuente: `claim`, `source_id`, `source_name`, `publication_date`. Conflictos entre fuentes NO se resuelven en silencio: se marcan y se enrutan a humano. Resúmenes en prosa libre quedan prohibidos.

## 2. Por qué importa

Después de agregar contenido de muchas fuentes (Kata 4), no preservar provenance hace imposible auditar. Un resumen plausible aluciinará sin que se note.

## 3. Ejemplo correcto — claims tipados con source_id

In [ ]:
import json

DOCS = {
    "doc-A": {
        "name": "Annual Report 2025", "date": "2025-12-01",
        "text": "ARR Q3 2025 = $12M. Headcount al cierre: 450 personas. Churn 8%.",
    },
    "doc-B": {
        "name": "Investor Deck", "date": "2025-09-15",
        "text": "ARR de cierre: $12.4M. ARR Q3 fue $12M.",
    },
    "doc-C": {
        "name": "HR Export", "date": "2026-01-10",
        "text": "Headcount cierre 2025: 462 personas (final ajuste).",
    },
}

CLAIM_TOOL = {
    "name": "submit_claims",
    "description": "Registra cada hecho con su fuente.",
    "input_schema": {
        "type": "object",
        "properties": {
            "claims": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "topic": {"type": "string"},
                        "value": {"type": "string"},
                        "source_id": {"type": "string"},
                    },
                    "required": ["topic","value","source_id"],
                },
            }
        },
        "required": ["claims"],
    },
}

def extract_claims(client, doc_id, doc):
    sys = "Extrae cada hecho atómico del documento como un claim independiente. NO resumas; NO omitas."
    user = f"DOC ID: {doc_id}\nNAME: {doc['name']}\nDATE: {doc['date']}\nTEXT:\n{doc['text']}"
    resp = client.messages.create(
        system=sys,
        messages=[{"role":"user","content": user}],
        tools=[CLAIM_TOOL],
        tool_choice={"type":"tool","name":"submit_claims"},
    )
    return next(b.input for b in resp.content if b.type == "tool_use")["claims"]

all_claims = []
for did, d in DOCS.items():
    cs = extract_claims(client, did, d)
    for c in cs:
        c["source_name"] = DOCS[c["source_id"]]["name"]
        c["source_date"] = DOCS[c["source_id"]]["date"]
    all_claims.extend(cs)
print(json.dumps(all_claims, indent=2, ensure_ascii=False))

### 3.1 Detección de conflictos sin resolver

In [ ]:
from collections import defaultdict
by_topic = defaultdict(list)
for c in all_claims:
    by_topic[c["topic"].lower()].append(c)

conflicts = []
agreements = []
for topic, items in by_topic.items():
    values = {i["value"] for i in items}
    if len(values) > 1:
        conflicts.append({"topic": topic, "items": items, "needs_human_review": True})
    else:
        agreements.append({"topic": topic, "value": items[0]["value"], "sources": [i["source_id"] for i in items]})

print("=== ACUERDOS ===")
print(json.dumps(agreements, indent=2, ensure_ascii=False))
print("\n=== CONFLICTOS (a humano) ===")
print(json.dumps(conflicts, indent=2, ensure_ascii=False))

Ningún conflicto se "resuelve" eligiendo. Headcount muestra `450` (doc-A 2025-12-01) vs `462` (doc-C 2026-01-10) — ambos quedan, con sources, listos para revisión humana (Kata 16).

## 4. Anti-patrón — resumen agregado en prosa

In [ ]:
big = "\n\n".join(f"=== {k} ({d['date']}) ===\n{d['text']}" for k, d in DOCS.items())
resp_bad = client.messages.create(
    system="Resume los hechos clave del corpus en un párrafo.",
    messages=[{"role":"user","content": big}],
)
print(next((b.text for b in resp_bad.content if b.type == "text"), "")[:600])

Síntomas observables del anti-patrón:

- El modelo elige un valor (450 o 462) sin avisar.
- Los números aparecen sin source_id; auditarlos requiere releer todo el corpus.
- El conflicto desaparece de la salida.

Cualquier downstream que actúe sobre ese resumen está construyendo encima de una decisión silenciosa.

## 5. Argumento de certificación

- **Invariante**: no hay claim sin source. Validable en código.
- **Conflictos preservados, no resueltos.** El agente reconoce que no le toca elegir.
- **Fechas y nombres de fuentes** acompañan al claim para que el humano juzgue (la fuente más reciente no siempre gana, pero el operador necesita verla).
- **Conexión con otros katas**: extracción tipada (Kata 5); fan-out con subagentes por doc (Kata 4); conflictos disparan Kata 16; correctitud numérica de cada claim respeta Kata 15.

## 6. Auto-evaluación

**1. ¿Qué hago si la fuente no tiene fecha?**

`source_date` se marca null o `unknown`. El claim sigue siendo válido pero el humano que resuelve un conflicto verá la ausencia de fecha como debilidad de la fuente.

**2. ¿Cuándo dos sources con el mismo `value` cuentan como "una sola confirmación" y cuándo como dos?**

Si comparten origen primario (mismo dataset, mismo reporte oficial copiado en dos lugares), una. Si son extracciones independientes con cadenas de evidencia distintas, dos. La heurística práctica: registrarlos por separado y dejar que el humano lo lea.

**3. ¿Qué prueba reintroduce el anti-patrón (prosa sin citas) y qué assert falla?**

Reemplazar la salida del extractor por prosa libre. Una aserción defensiva: para cada claim final, `assert "source_id" in claim and claim["source_id"] in known_doc_ids`. Si el modelo emite un claim sin source_id (o con uno fabricado), el assert falla y el pipeline rechaza la respuesta.